In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Define the inner 'data' structure
data_schema = StructType([
    StructField("sym", StringType(), True),
    StructField("ltt", StringType(), True),
    StructField("ltp", StringType(), True), # Often comes as string in JSON, cast later
    StructField("vol", StringType(), True)
])

# Define the full nested structure
json_schema = StructType([
    StructField("response", StructType([
        StructField("data", data_schema, True),
        StructField("streaming_type", StringType(), True)
    ]), True)
])

df = (
    spark.readStream
    .format("cloudFiles")
    .schema(json_schema)
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/main/default/vol2/2885_NSE_schemaLocation")
    .load("/Volumes/main/default/vol2/2885_NSE/")
)

In [0]:
# Flatten and rename columns
main_df = df.select(
    col("response.data.sym").alias("symbol"),
    to_timestamp(col("response.data.ltt"), "d MMM yyyy, hh:mm:ss a").alias("trade_time"),
    col("response.data.ltp").cast("double").alias("close"),
    col("response.data.vol").cast("long").alias("volume")
)

In [0]:
grouped_1m_df = main_df.withWatermark("trade_time", "2 seconds").groupBy(window(col("trade_time"), "1 minute"), col("symbol")).agg(
    first("close").alias("open"),
max("close").alias("high"),
min("close").alias("low"),
last("close").alias("close"),
sum("volume").alias("volume")
).select(to_timestamp("window.start").alias("trade_time"), "symbol", "open", "high", "low", "close", "volume")

In [0]:
query = (
    grouped_1m_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/main/default/vol2/2885_NSE_checkpoint")
    .trigger(processingTime="5 seconds")
    .toTable("main.default.samco_1m_candles_RELIANCE")
)